In [2]:
# A/B Test: Email Campaign Performance Simulation and Analysis

import pandas as pd
import numpy as np
from faker import Faker
import random

# Initialize faker
fake = Faker()
np.random.seed(42)

# 1. Simulate A/B Test data for 2 groups: A (control), B (test)
n_users = 500
ab_data = []

for i in range(1, n_users + 1):
    group = random.choice(['A', 'B'])
    open_email = np.random.binomial(1, 0.35 if group == 'A' else 0.42)
    click_email = np.random.binomial(1, 0.18 if group == 'A' else 0.25) if open_email else 0
    purchase = np.random.binomial(1, 0.07 if group == 'A' else 0.11) if click_email else 0

    ab_data.append({
        'user_id': i,
        'group': group,
        'open': open_email,
        'click': click_email,
        'purchase': purchase
    })

# Convert to DataFrame
ab_df = pd.DataFrame(ab_data)

# Quick overview
print(ab_df.head())
print("\nGroup counts:\n", ab_df['group'].value_counts())

# Save for BI tools (e.g. Tableau / Power BI)
ab_df.to_csv("ab_campaign_results.csv", index=False)

# Aggregate metrics per group
summary = ab_df.groupby("group").agg(
    total_users=('user_id', 'count'),
    open_rate=('open', 'mean'),
    click_rate=('click', 'mean'),
    purchase_rate=('purchase', 'mean')
).reset_index()

# Lift calculations
control = summary.loc[summary['group'] == 'A']
test = summary.loc[summary['group'] == 'B']
summary['open_lift'] = summary['open_rate'] - float(control['open_rate'].iloc[0])
summary['click_lift'] = summary['click_rate'] - float(control['click_rate'].iloc[0])
summary['purchase_lift'] = summary['purchase_rate'] - float(control['purchase_rate'].iloc[0])

print("\nA/B Test Summary:\n")
print(summary)

# Optional: Save summary to CSV
summary.to_csv("ab_summary_metrics.csv", index=False)

   user_id group  open  click  purchase
0        1     A     0      0         0
1        2     B     1      0         0
2        3     B     1      0         0
3        4     B     0      0         0
4        5     A     0      0         0

Group counts:
 group
B    259
A    241
Name: count, dtype: int64

A/B Test Summary:

  group  total_users  open_rate  click_rate  purchase_rate  open_lift  \
0     A          241   0.311203    0.078838       0.012448   0.000000   
1     B          259   0.389961    0.088803       0.011583   0.078758   

   click_lift  purchase_lift  
0    0.000000       0.000000  
1    0.009965      -0.000865  
